# CSST102: Predicting Persistence of Human Rabies Deaths Across CALABARZON Municipalities (2025 to 2026)
**Section:** BSCS 3-A | **Group:** Group 6  
**Members:** John Kurt S. Larioza (Leader), Airelle Lambert E. Ramos, Charles Lawrence R. Ferrer  
**Instructor:** Rolan Sotomayor  
**Data Source:** DOH CHD IV-CALABARZON, *2026 Regional Updates on the Rabies Prevention and Control Program* (Slides 9–11, 2025 & 2026 as of Sept 5, 2026; context Slides 3, 6, 14)

In [1]:
# Week 1 & Week 2: Load Dataset, Inspect Missing/Duplicates, Define Target & Features
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

df = pd.read_csv('calabarzon_municipal_rabies_deaths_2025_2026.csv')
print('Dataset shape:', df.shape)
print('Missing values:', df.isnull().sum().to_dict())
print('Duplicates:', df.duplicated().sum())
print('Total 2025 deaths:', df['deaths_2025'].sum(), '| Total 2026 deaths:', df['deaths_2026'].sum())
print('Target (target_death_2026) distribution:', df['target_death_2026'].value_counts().to_dict())
df.head(10)

In [2]:
# Week 3: Preprocessing Pipeline, Stratified 70/30 Split & Baseline Logistic Regression
# Exclude deaths_2026 and target_death_2026 from input features to avoid data leakage
features = ['province', 'deaths_2025', 'is_city', 'high_2025']
X = df[features]
y = df['target_death_2026']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(f'Train rows: {len(X_train)} | Test rows: {len(X_test)}')

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['province']),
    ('num', StandardScaler(), ['deaths_2025']),
    ('pass', 'passthrough', ['is_city', 'high_2025'])
])

lr_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]

print('Baseline Logistic Regression Train Acc:', round(accuracy_score(y_train, lr_pipe.predict(X_train)), 3))
print('Baseline Logistic Regression Test Acc:', round(accuracy_score(y_test, y_pred_lr), 3))

In [3]:
# Week 4: Model Evaluation & Comparison (Dummy, Logistic Regression, Random Forest + 5-Fold CV)
dummy_pipe = Pipeline([('prep', preprocessor), ('clf', DummyClassifier(strategy='most_frequent'))]).fit(X_train, y_train)
rf_pipe = Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))]).fit(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in [('Dummy', dummy_pipe), ('LogReg Baseline', lr_pipe), ('Random Forest Baseline', rf_pipe)]:
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc').mean()
    print(f'{name:22s} | Acc: {accuracy_score(y_test, preds):.3f} | Prec: {precision_score(y_test, preds, zero_division=0):.3f} | '
          f'Rec: {recall_score(y_test, preds, zero_division=0):.3f} | F1: {f1_score(y_test, preds, zero_division=0):.3f} | '
          f'Test AUC: {roc_auc_score(y_test, probs):.3f} | 5-Fold CV AUC: {cv_auc:.3f}')

In [4]:
# Week 5: Hyperparameter Tuning (GridSearchCV), Error Analysis & Final Metrics
lr_grid = GridSearchCV(
    Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    param_grid={'clf__C': [0.01, 0.1, 1.0, 10.0]},
    scoring='roc_auc', cv=cv
).fit(X_train, y_train)

rf_grid = GridSearchCV(
    Pipeline([('prep', preprocessor), ('clf', RandomForestClassifier(random_state=42))]),
    param_grid={'clf__max_depth': [2, 3, 5], 'clf__min_samples_leaf': [2, 4], 'clf__n_estimators': [100]},
    scoring='roc_auc', cv=cv
).fit(X_train, y_train)

print('Best LR params:', lr_grid.best_params_, '| Best CV AUC:', round(lr_grid.best_score_, 3))
print('Best RF params:', rf_grid.best_params_, '| Best CV AUC:', round(rf_grid.best_score_, 3))

y_pred_rf_tuned = rf_grid.predict(X_test)
y_prob_rf_tuned = rf_grid.predict_proba(X_test)[:, 1]

print('\nTuned Random Forest Test Accuracy:', round(accuracy_score(y_test, y_pred_rf_tuned), 3))
print('Tuned Random Forest Test F1:', round(f1_score(y_test, y_pred_rf_tuned), 3))
print('Tuned Random Forest Test ROC-AUC:', round(roc_auc_score(y_test, y_prob_rf_tuned), 3))
print('\nConfusion Matrix (Tuned RF):\n', confusion_matrix(y_test, y_pred_rf_tuned))

test_results = df.loc[X_test.index, ['municipality', 'province', 'deaths_2025', 'target_death_2026']].copy()
test_results['pred_rf_tuned'] = y_pred_rf_tuned
test_results['prob_rf_tuned'] = np.round(y_prob_rf_tuned, 3)
print('\nTest Set Predictions & Errors:')
print(test_results.to_string(index=False))